# 319 Phase 2 — XGBoost Final Pool 외부 검증 (chan_browser n=324)

**참조**: `services/ai/train/model_experiment/xgboost_gyeom_final_pool.ipynb` (Phase 1, 보존 read-only)

## 본 phase scope (단순화)

**(a) eval-only**: Phase 1 `model.joblib` 으로 chan_browser 324 trials 외부 검증.
chan 단독 ROC AUC + threshold 0.40/0.75 confusion matrix + recall@0.40 / precision@0.75 + allow_FN_op 산출.

진단 figure / ADR-017 update / 후속 input 작성 등 부수 산출은 §2 결과 보고 후 분석 chat 합의로 별도 결정 — 본 phase 폐기.

## Phase 1 reference (보존)

- model: `services/ai/train/model_joblib/xgboost_gyeom_final_pool/model.joblib`
- input_features: `mouse_jerk_mean`, `mouse_max_speed_px_per_ms`
- meta: `production_ready=false / do_not_load_to_production=true / 5 blocking_flags`
- §5b lv3_kde recall@0.40 = 0.00 / allow_FN_op = 0.92 FAIL — chan 결과와 직접 비교 reference

## 평가 set

`services/ai/data/behavior_chan/behavior/trial_*.json` 324 trials (label balanced — macro 164 / human 160).
trial_loader: `list_trials_by_group("chan_browser")`.

## 산출물

본 phase 산출물 = 본 노트북 cells (§0 + §1 + §2). meta.json / model.joblib 신규 생성 X (Phase 1 4 artifacts 그대로 사용).

## Decision lock 요약 (plan)

- #1: 위치 = `behavior_chan/behavior/`, range 1~324, group `chan_browser`
- #2: (a) eval-only 채택
- #3: 신규 노트북 (본 파일)
- #4: §2 ROC AUC + cm + allow_FN_op (§3 진단 figure 폐기)
- #5: 좌표계 정합 — Round 4 §11 분포 추정 reference 보존, ADR update markdown 산출 폐기

## §0 Setup — Phase 1 reference 로드

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

SEED = 42
MODEL_NAME = "xgboost_gyeom_final_pool"
LABEL_MAPPING = {"human": 0, "macro": 1}

ROOT = Path.cwd()
if ROOT.name != "model_experiment":
    raise Exception(f"작업 디렉토리는 model_experiment 여야 함 (현재: {ROOT.name})")

# Phase 1 4 artifacts 경로
PHASE1_DIR = Path("../model_joblib") / MODEL_NAME
MODEL_PATH = PHASE1_DIR / "model.joblib"
META_PATH = PHASE1_DIR / "meta.json"
METRICS_PATH = PHASE1_DIR / "metrics.json"
INPUT_FEATURES_PATH = PHASE1_DIR / "input_features.json"

# chan_browser 그룹 (services/ai/train/EDA/trial_loader.py)
sys.path.insert(0, str(Path("../EDA").resolve()))
from trial_loader import list_trials_by_group, CHAN_BROWSER_RANGE

# input_features 로드 (Phase 1 final pool 2)
with INPUT_FEATURES_PATH.open("r", encoding="utf-8") as f:
    INPUT_FEATURES = json.load(f)

print("ROOT:", ROOT)
print("PHASE1_DIR:", PHASE1_DIR)
print("MODEL_PATH exists:", MODEL_PATH.exists())
print("CHAN_BROWSER_RANGE:", CHAN_BROWSER_RANGE)
print("INPUT_FEATURES (Phase 1):", INPUT_FEATURES)